<a href="https://colab.research.google.com/github/BenayaL/Cloud_Computing_Project/blob/main/%D7%A2%D7%95%D7%AA%D7%A7_%D7%A9%D7%9C_WombatCombatPrj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title #Imports

!pip install -q PyPDF2 gdown ipywidgets pandas matplotlib nltk

import os
import re
import glob
import html
import json
import datetime
import traceback
import firebase_admin
import gdown
import PyPDF2
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from google.colab import userdata
from firebase_admin import credentials, db
from nltk.stem import PorterStemmer
from IPython.display import display, HTML, clear_output
from ipywidgets import Layout, VBox, HBox, Output, ButtonStyle


In [ ]:
#@title #Configuration

FOLDER_ID = "1zhHkG9eYIy6__Tnt7V-o_7293m3mz80W"
FOLDER_URL = f"https://drive.google.com/drive/folders/{FOLDER_ID}"

# External cloud services configuration
# Gemini is used for the RAG answer. Firebase is used as the cloud database.
USE_GEMINI = True
USE_FIREBASE = True

def get_secret(secret_name):
    try:
        return userdata.get(secret_name)
    except Exception:
        return None

GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
FIREBASE_DATABASE_URL = get_secret("FIREBASE_DATABASE_URL")
FIREBASE_SERVICE_ACCOUNT_JSON = get_secret("FIREBASE_SERVICE_ACCOUNT")

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print("Gemini API key loaded successfully.")
else:
    USE_GEMINI = False
    print("Gemini API key was not found. Gemini is disabled.")

if not FIREBASE_DATABASE_URL:
    USE_FIREBASE = False
    print("Firebase database URL was not found. Firebase is disabled.")

if not FIREBASE_SERVICE_ACCOUNT_JSON:
    USE_FIREBASE = False
    print("Firebase service account was not found. Firebase is disabled.")

ARTICLE_INFO = {}

# Stop Words Configuration
STOP_WORDS = {
    "a", "an", "the", "and", "or", "in", "on", "at", "of", "with", "by", "as",
    "to", "their", "under", "for", "is", "are", "was", "were", "be", "been", "being",
    "this", "that", "these", "those", "from", "into", "it", "its", "we", "our", "can",
    "may", "also", "using", "use", "used", "than", "then", "there", "such", "between"
}

DOMAIN_STOP_WORDS = {"basil", "plant", "plants", "ocimum", "basilicum", "l"}
STOP_WORDS.update(DOMAIN_STOP_WORDS)

# Target Words Configuration
IMPORTANT_TERMS = [
    "disease", "mildew", "fungal", "fungus", "pathogen",
    "infection", "symptoms", "leaf", "leaves", "yellowing",
    "spots", "humidity", "moisture", "soil", "water",
    "stress", "irrigation", "temperature", "light", "growth",
    "yield", "quality", "nutrition", "fertilization", "management",
    "rhizobacteria", "pgpr", "antioxidant", "organic", "soilless"
]

stemmer = PorterStemmer()


Gemini API key loaded successfully.


In [ ]:
#@title #Firebase Service

FIREBASE_READY = False
FIREBASE_DB = None

def init_firebase():
    """Initialize Firebase Realtime Database using Colab Secrets."""
    global FIREBASE_READY, FIREBASE_DB

    try:
        import firebase_admin
        from firebase_admin import credentials, db

        if not FIREBASE_DATABASE_URL:
            print("Firebase database URL is missing. Add FIREBASE_DATABASE_URL to Colab Secrets.")
            return False

        if not FIREBASE_SERVICE_ACCOUNT_JSON:
            print("Firebase service account JSON is missing. Add FIREBASE_SERVICE_ACCOUNT to Colab Secrets.")
            return False

        firebase_config = json.loads(FIREBASE_SERVICE_ACCOUNT_JSON)

        if not firebase_admin._apps:
            cred = credentials.Certificate(firebase_config)
            firebase_admin.initialize_app(cred, {
                "databaseURL": FIREBASE_DATABASE_URL
            })

        FIREBASE_DB = db
        FIREBASE_READY = True
        print("✅ Firebase connected successfully")
        return True

    except Exception as e:
        FIREBASE_READY = False
        print(f"Firebase initialization failed: {e}")
        return False


def firebase_save_index(index_rows):
    """Save the inverted index to Firebase."""
    if not FIREBASE_READY:
        return False

    try:
        rows_as_dict = index_rows.to_dict(orient="records")
        FIREBASE_DB.reference("index").set(rows_as_dict)
        return True
    except Exception as e:
        print(f"Could not save index to Firebase: {e}")
        return False


def firebase_save_articles_metadata(articles):
    """Save only article metadata to Firebase, not the full PDF text."""
    if not FIREBASE_READY:
        return False

    try:
        metadata = {}

        for article in articles:
            metadata[article["id"]] = {
                "title": article.get("title", ""),
                "authors": article.get("authors", ""),
                "url": article.get("url", ""),
                "file_name": article.get("file_name", "")
            }

        FIREBASE_DB.reference("articles").set(metadata)
        return True
    except Exception as e:
        print(f"Could not save articles metadata to Firebase: {e}")
        return False


def firebase_save_search_history(query, results_count):
    """Save user search history to Firebase."""
    if not FIREBASE_READY:
        return False

    try:
        item = {
            "query": query,
            "results_count": results_count,
            "timestamp": datetime.datetime.now().isoformat()
        }

        FIREBASE_DB.reference("search_history").push(item)
        return True
    except Exception as e:
        print(f"Could not save search history to Firebase: {e}")
        return False


def firebase_get_latest_sensor_reading():
    """Read the newest sensor reading from Firebase, if one exists."""
    if not FIREBASE_READY:
        return None

    try:
        data = FIREBASE_DB.reference("sensor_readings").order_by_key().limit_to_last(1).get()

        if not data:
            return None

        last_key = list(data.keys())[-1]
        return data[last_key]

    except Exception as e:
        print(f"Could not read sensor data from Firebase: {e}")
        return None


if USE_FIREBASE:
    init_firebase()
else:
    print("Firebase is disabled. Running locally only.")

✅ Firebase connected successfully


In [ ]:
#@title #Gemini API Service

GEMINI_MODEL = None

def init_gemini():
    """Initialize Gemini from Colab Secrets or from GEMINI_API_KEY."""
    global GEMINI_MODEL

    try:
        import google.generativeai as genai

        api_key = GEMINI_API_KEY

        if not api_key:
            try:
                from google.colab import userdata
                api_key = userdata.get("GEMINI_API_KEY")
            except Exception:
                api_key = None

        if not api_key:
            print("Gemini API key was not found. Add GEMINI_API_KEY in Colab Secrets or paste it in Configuration.")
            return False

        genai.configure(api_key=api_key)
        GEMINI_MODEL = genai.GenerativeModel("gemini-2.5-flash-lite")
        print("✅ Gemini connected successfully")
        return True

    except Exception as e:
        print(f"Gemini initialization failed: {e}")
        return False


if USE_GEMINI:
    init_gemini()

✅ Gemini connected successfully


In [ ]:
#@title #Data loading

folder_path = "/content/drive_pdfs"
os.makedirs(folder_path, exist_ok=True)

print(f"Fetching files from Google Drive: {FOLDER_URL}...")
gdown.download_folder(url=FOLDER_URL, output=folder_path, quiet=False, use_cookies=False)
print("Download complete. Starting extraction...\n")


def clean_filename_as_title(file_name):
    """
    Convert a PDF file name into a readable temporary title.
    """
    base = os.path.splitext(file_name)[0]
    base = base.replace("_", " ").replace("-", " ")
    return " ".join(base.split()).title()


def extract_text_from_pdf(pdf_path):
    """
    Extract text from a PDF file using PyPDF2.
    """
    extracted_text = ""

    try:
        with open(pdf_path, "rb") as file:
            pdf_reader = PyPDF2.PdfReader(file)

            for page in pdf_reader.pages:
                page_text = page.extract_text()

                if page_text:
                    extracted_text += page_text + " "

    except Exception as e:
        print(f"Could not read PDF: {os.path.basename(pdf_path)} | Error: {e}")

    return extracted_text.strip()


# Find and load all PDFs
pdf_files = sorted(glob.glob(os.path.join(folder_path, "**", "*.pdf"), recursive=True))
ARTICLES = []

if not pdf_files:
    print(f"Error: No PDF files found in {folder_path}")

else:
    for doc_id, path in enumerate(pdf_files, start=1):
        file_name = os.path.basename(path)
        extracted_text = extract_text_from_pdf(path)

        if not extracted_text:
            print(f"⚠️ No readable text extracted from {file_name}. It may be scanned or image-based.")
            continue

        info = ARTICLE_INFO.get(file_name, {})

        # **look into how to parse into each file and extract the information to fill these data cells**
        article = {
            "id": f"doc{doc_id}",
            "title": info.get("title", clean_filename_as_title(file_name)),
            "authors": info.get("authors", "Add real authors in ARTICLE_INFO or in the report"),
            "url": info.get("url", "Add DOI/URL in ARTICLE_INFO or in the report"),
            "file_name": file_name,
            "text": extracted_text
        }

        ARTICLES.append(article)
        print(f"Successfully loaded {file_name} as Document {doc_id}")

    print(f"\n✅ All {len(ARTICLES)} documents loaded successfully!")

print(f"\nTotal readable academic documents loaded: {len(ARTICLES)}")


Fetching files from Google Drive: https://drive.google.com/drive/folders/1zhHkG9eYIy6__Tnt7V-o_7293m3mz80W...


Retrieving folder contents


Processing file 1NlB1A_X9bus2pEyM8M1gtBoZTZPmANSg 1-s2.0-S0570178316300288-main.pdf
Processing file 1hDZwSBDig6htKY0x-bOm7847Hb7vNm19 editor1,+TAIE.pdf
Processing file 1DqG3R6hzdPkYx7meNja1fkdi59GJbHj9 IJOEAR-MAY-2016-10.pdf
Processing file 1MVh50P4GkZ6uSC9LZi2hoDyozTcu0mX9 pdis.1997.81.2.124.pdf
Processing file 1y9DJRVa2PouZyh1ErmNJFi4YLUMgUqxa Scientifica - 2020 - Al-Huqail - Effects of Climate Temperature and Water Stress on Plant Growth and Accumulation of.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1NlB1A_X9bus2pEyM8M1gtBoZTZPmANSg
To: /content/drive_pdfs/1-s2.0-S0570178316300288-main.pdf
100%|██████████| 697k/697k [00:00<00:00, 94.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hDZwSBDig6htKY0x-bOm7847Hb7vNm19
To: /content/drive_pdfs/editor1,+TAIE.pdf
100%|██████████| 326k/326k [00:00<00:00, 27.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1DqG3R6hzdPkYx7meNja1fkdi59GJbHj9
To: /content/drive_pdfs/IJOEAR-MAY-2016-10.pdf
100%|██████████| 1.31M/1.31M [00:00<00:00, 57.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MVh50P4GkZ6uSC9LZi2hoDyozTcu0mX9
To: /content/drive_pdfs/pdis.1997.81.2.124.pdf
100%|██████████| 628k/628k [00:00<00:00, 35.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1y9DJRVa2PouZyh1ErmNJFi4YLUMgUqxa
To: /content/drive_pdfs/Scientifica - 2020 - Al-Huqail - Effects 

Download complete. Starting extraction...

Successfully loaded 1-s2.0-S0570178316300288-main.pdf as Document 1
Successfully loaded IJOEAR-MAY-2016-10.pdf as Document 2
Successfully loaded Scientifica - 2020 - Al-Huqail - Effects of Climate Temperature and Water Stress on Plant Growth and Accumulation of.pdf as Document 3
Successfully loaded editor1,+TAIE.pdf as Document 4
Successfully loaded pdis.1997.81.2.124.pdf as Document 5

✅ All 5 documents loaded successfully!

Total readable academic documents loaded: 5


In [ ]:
#@title #Text pre-processing

def tokenize(text):
    """
    Split text into lowercase English word tokens.
    Example: "Leaf spots!" -> ["leaf", "spots"]
    """
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())


def preprocess_text(text):
    """
    Full text processing pipeline:
    1. Tokenization
    2. Stop words removal
    3. Stemming with PorterStemmer
    """
    tokens = tokenize(text)
    processed_terms = []

    for token in tokens:
        if token not in STOP_WORDS:
            processed_terms.append(stemmer.stem(token))

    return processed_terms


In [ ]:
#@title #Index Service

def build_inverted_index(articles, important_terms):
    """
    Build an inverted index only for the important academic terms.
    This keeps the index clear and aligned with the homework requirement.
    """
    stemmed_targets = {
        original_term: stemmer.stem(original_term.lower())
        for original_term in important_terms
    }

    inverted_index = {
        stemmed_term: set()
        for stemmed_term in stemmed_targets.values()
    }

    for article in articles:
        doc_id = article["id"]
        terms_in_doc = set(preprocess_text(article["text"]))

        for original_term, stemmed_term in stemmed_targets.items():
            if stemmed_term in terms_in_doc:
                inverted_index[stemmed_term].add(doc_id)

    return inverted_index, stemmed_targets


def build_index_dataframe(inverted_index, stemmed_targets):
    """
    Create the exact table required by the homework:
    term | DocIDs
    """
    rows = []

    for original_term, stemmed_term in stemmed_targets.items():
        doc_ids = sorted(inverted_index.get(stemmed_term, set()))
        rows.append({
            "term": original_term,
            "DocIDs": ", ".join(doc_ids) if doc_ids else "None"
        })

    return pd.DataFrame(rows).sort_values("term").reset_index(drop=True)


INVERTED_INDEX, STEMMED_TARGETS = build_inverted_index(ARTICLES, IMPORTANT_TERMS)
df_index = build_index_dataframe(INVERTED_INDEX, STEMMED_TARGETS)

# Save the index to Firebase when the connection is ready.
# The UI no longer displays the full table automatically.
firebase_save_index(df_index)
firebase_save_articles_metadata(ARTICLES)


True

In [ ]:
#@title #Query Service

def search_articles(query, articles, inverted_index):
    """
    Search academic articles using the inverted index.

    Returns:
    - results: list of tuples (article_dict, score, matched_terms)
    - query_terms: normalized query terms
    """
    query_terms = preprocess_text(query)
    scores = {}
    matched_terms_by_doc = {}

    for term in query_terms:
        if term in inverted_index:
            matching_doc_ids = inverted_index[term]

            for doc_id in matching_doc_ids:
                scores[doc_id] = scores.get(doc_id, 0) + 1
                matched_terms_by_doc.setdefault(doc_id, set()).add(term)

    sorted_doc_ids = sorted(scores.keys(), key=lambda doc_id: scores[doc_id], reverse=True)

    results = []
    for doc_id in sorted_doc_ids:
        article = next((a for a in articles if a["id"] == doc_id), None)
        if article:
            results.append((article, scores[doc_id], sorted(matched_terms_by_doc.get(doc_id, []))))

    return results, query_terms


def build_context_from_results(results, max_chars_per_doc=1200):
    """
    Build a context block from retrieved academic articles.
    This is the Retrieval part of RAG.
    """
    if not results:
        return ""

    context_parts = []

    for rank, (article, score, matched_terms) in enumerate(results, start=1):
        text = article["text"][:max_chars_per_doc]
        context_parts.append(
            f"Source {rank}: {article['title']} ({article['id']})\n"
            f"Matched terms: {', '.join(matched_terms)}\n"
            f"Text excerpt:\n{text}\n"
        )

    return "\n---\n".join(context_parts)


In [ ]:
#@title #RAG Service

def simple_rag_answer(query, results):
    """
    Simple RAG-style answer that works without external AI API.
    It uses the retrieved article titles and matched terms to generate a practical response.
    """
    if not results:
        return (
            "No relevant academic article was found for this query. "
            "Try searching for terms such as mildew, humidity, moisture, leaf, disease, or temperature."
        )

    top_article, top_score, matched_terms = results[0]
    matched_text = ", ".join(matched_terms) if matched_terms else "the query terms"

    return (
        f"The query is most related to <b>{html.escape(top_article['title'])}</b>. "
        f"The system matched the academic terms: <b>{html.escape(matched_text)}</b>. "
        f"Based on the retrieved context, the basil plant should be checked for visible leaf symptoms, "
        f"soil moisture, humidity, temperature, and general disease indicators. "
        f"For a real recommendation, the retrieved article context should be combined with the current IoT readings."
    )


def gemini_rag_answer(query, results):
    """
    Optional real AI answer using Gemini.
    This is disabled by default to avoid API/key/model errors during presentation.
    """
    try:
        import google.generativeai as genai

        context = build_context_from_results(results)
        prompt = f"""
        You are an academic assistant inside a cloud computing RAG prototype for basil plant monitoring.

        Your role:
        The user asks a question about basil plants, diseases, humidity, water stress, temperature,
        growth conditions, or plant health.
        You must answer using ONLY the retrieved academic context below.
        Do NOT use outside knowledge.
        Do NOT invent facts that are not supported by the retrieved context.

        Important behavior:
        - If the retrieved context directly answers the question, give a clear academic answer.
        - If the retrieved context is only partially related, clearly say:
          "The retrieved articles provide related evidence, but not a full direct answer."
        - If the user asks about mildew but the context mentions other fungal diseases, explain that the articles discuss fungal disease risk generally, but may not directly discuss mildew.
        - Mention specific diseases, symptoms, or environmental conditions only if they appear in the retrieved context.
        - Keep the answer useful for a basil monitoring dashboard.

        Required answer format:
        1. Direct answer: 1-2 sentences.
        2. Evidence from retrieved articles: 2-3 sentences.
        3. Practical monitoring implication: 1 sentence.
        4. Limitation: 1 sentence if the context is partial.

        Academic context:
        {context}

        User question:
        {query}
        """

        if GEMINI_MODEL is None:
            return simple_rag_answer(query, results)

        response = GEMINI_MODEL.generate_content(prompt)
        return response.text

    except Exception as e:
        return f"AI connection failed, using simple RAG is recommended. Error: {e}"


def rag_answer(query, results):
    """
    Main RAG function used by the UI.
    Uses Gemini only if USE_GEMINI=True, otherwise uses simple RAG.
    """
    if USE_GEMINI:
        return gemini_rag_answer(query, results)

    return simple_rag_answer(query, results)


In [ ]:
#@title #Plant Health Logic

def get_plant_health_status():
    return {
        "status": "Not implemented yet",
        "message": "Sensor-based plant health logic will be added in a future iteration."
    }

In [ ]:
#@title #CSS Styles

APP_CSS = """
<style>
.wb-app {
    font-family: Arial, sans-serif;
    direction: ltr;
    background: linear-gradient(135deg, #edf8ef 0%, #fffaf0 100%);
    border-radius: 24px;
    padding: 22px;
    border: 1px solid #dbeadf;
}
.wb-header {
    background: linear-gradient(135deg, #2e7d5b, #69b578);
    color: white;
    border-radius: 22px;
    padding: 22px 26px;
    margin-bottom: 18px;
    box-shadow: 0 10px 24px rgba(46, 125, 91, 0.20);
}
.wb-header h1 { margin: 0; font-size: 30px; }
.wb-header p { margin: 8px 0 0 0; opacity: 0.92; }
.card {
    background: white;
    border: 1px solid #e3eee6;
    border-radius: 18px;
    padding: 18px;
    margin: 12px 0;
    box-shadow: 0 8px 18px rgba(0,0,0,0.045);
}
.section-title { color: #255f46; margin: 0 0 10px 0; font-size: 22px; }
.muted { color: #6c7a70; font-size: 14px; }
.metric-grid {
    display: grid;
    grid-template-columns: repeat(4, minmax(130px, 1fr));
    gap: 12px;
    margin: 12px 0;
}
.metric {
    background: #fbfffc;
    border: 1px solid #dceee2;
    border-radius: 16px;
    padding: 14px;
}
.metric .label { color: #6c7a70; font-size: 13px; }
.metric .value { color: #193d2e; font-weight: 800; font-size: 26px; margin-top: 6px; }
.badge {
    display: inline-block;
    padding: 5px 10px;
    border-radius: 999px;
    font-weight: bold;
    font-size: 13px;
}
.green { background: #e5f8ec; color: #1f7a4d; border: 1px solid #bce6ca; }
.orange { background: #fff4df; color: #a96500; border: 1px solid #f1d39b; }
.red { background: #ffe8e8; color: #b42323; border: 1px solid #f3b4b4; }
.result-card {
    background: #fcfffd;
    border-left: 5px solid #2e7d5b;
    border-radius: 14px;
    padding: 14px;
    margin: 10px 0;
}
.rag-box {
    background: #eefaf2;
    border: 1px solid #ccebd6;
    border-radius: 16px;
    padding: 16px;
    margin-top: 14px;
}
.task {
    display: flex;
    align-items: center;
    gap: 10px;
    padding: 10px 12px;
    background: #fbfffc;
    border: 1px solid #e1eee5;
    border-radius: 14px;
    margin: 8px 0;
}

/* Rounded buttons for ipywidgets */
.widget-button {
    border-radius: 999px !important;
    border: 1px solid #dceee2 !important;
    font-weight: 700 !important;
    box-shadow: 0 6px 14px rgba(46, 125, 91, 0.12) !important;
}

/* Hover effect */
.widget-button:hover {
    transform: translateY(-1px);
    box-shadow: 0 8px 18px rgba(46, 125, 91, 0.18) !important;
}

/* Search input nicer */
.widget-text input {
    border-radius: 999px !important;
    border: 1px solid #dceee2 !important;
    padding: 8px 14px !important;
}


</style>
"""

display(HTML(APP_CSS))


def badge(text, tone):
    """Create a small colored HTML badge."""
    return f'<span class="badge {tone}">{html.escape(str(text))}</span>'


def metric_card(label, value, unit=""):
    """Create a reusable dashboard metric card."""
    return f"""
    <div class="metric">
        <div class="label">{html.escape(str(label))}</div>
        <div class="value">{html.escape(str(value))}{html.escape(str(unit))}</div>
    </div>
    """


In [ ]:
#@title #UI Screens

main_output = Output()


def render_dashboard_screen():
    """Dashboard screen based on real Firebase data when available."""
    with main_output:
        clear_output()
        display(HTML(APP_CSS))

        reading = firebase_get_latest_sensor_reading()

        if not reading:
            display(HTML(f"""
            <div class="card">
                <h2 class="section-title">📊 Basil Health Dashboard</h2>
                <p class="muted">
                    No real sensor readings are connected yet. This is intentional for now,
                    because the sensor part is still undefined by the team.
                </p>
                <p>{badge("Waiting for Firebase sensor data", "orange")}</p>
            </div>
            """))
            return

        score = calculate_health_score(reading)
        label, tone = health_label(score)
        updated_at = reading.get("updated_at", "unknown")

        display(HTML(f"""
        <div class="card">
            <h2 class="section-title">📊 Basil Health Dashboard</h2>
            <p class="muted">Latest real reading loaded from Firebase.</p>
            <div class="metric-grid">
                {metric_card("Soil moisture", reading.get("soil_moisture", "—"), "%")}
                {metric_card("Temperature", reading.get("temperature", "—"), "°C")}
                {metric_card("Air humidity", reading.get("air_humidity", "—"), "%")}
                {metric_card("Light", reading.get("light_level", "—"), " lux")}
            </div>
            <p><b>Updated at:</b> {html.escape(str(updated_at))}</p>
            <p><b>Health status:</b> {badge(label, tone)}</p>
            <p><b>Recommendation:</b> {html.escape(dashboard_recommendation(reading))}</p>
        </div>
        """))

# ---------- Upload screen ----------
upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="Choose image")
analyze_image_button = widgets.Button(
    description="Analyze Image",
    style=ButtonStyle(button_color="#2e7d5b"),
    layout=Layout(width="170px")
)
upload_output = Output()


def get_uploaded_file_name(file_upload_widget):
    """Return uploaded image file name across different ipywidgets versions."""
    value = file_upload_widget.value

    if isinstance(value, (tuple, list)) and len(value) > 0:
        return value[0].get("name", "uploaded image")

    if isinstance(value, dict) and len(value) > 0:
        return list(value.keys())[0]

    return None


def on_analyze_image_clicked(button):
    """
    Placeholder image analysis.
    Later this can call a real CV model for basil leaf symptoms.
    """
    with upload_output:
        clear_output()
        display(HTML(APP_CSS))
        file_name = get_uploaded_file_name(upload_widget)

        if not file_name:
            display(HTML('<div class="card">Please upload a basil image first.</div>'))
            return

        display(HTML(f"""
        <div class="result-card">
            <h3>Image received: {html.escape(file_name)}</h3>
            <p>Prototype result: {badge("Pending real AI model", "orange")}</p>
            <p class="muted">
                In the final version, this image can be analyzed for yellow leaves,
                spots, mildew, or other visible basil symptoms.
            </p>
        </div>
        """))


analyze_image_button.on_click(on_analyze_image_clicked)


def render_upload_screen():
    """Upload basil image screen."""
    with main_output:
        clear_output()
        display(HTML(APP_CSS))
        display(HTML("""
        <div class="card">
            <h2 class="section-title">📷 Upload Basil Image</h2>
            <p class="muted">
                Upload a basil plant image. The current prototype keeps the UI flow,
                and a real image model can be connected later.
            </p>
        </div>
        """))
        display(VBox([upload_widget, analyze_image_button, upload_output]))


# ---------- IoT screen ----------
def render_iot_screen():
    """IoT screen intentionally left empty for now."""
    with main_output:
        clear_output()
        display(HTML(APP_CSS))
        display(HTML(f"""
        <div class="card">
            <h2 class="section-title">📡 Sensors / IoT</h2>
            <p class="muted">
                This section is intentionally empty in the current iteration.
                The team has not yet decided which sensors will be used or what exact fields they will send.
            </p>
            <p>{badge("To be defined in the next iteration", "orange")}</p>
        </div>
        """))

# ---------- Search / RAG screen ----------
search_input = widgets.Text(
    value="basil mildew humidity",
    placeholder="Search: mildew, humidity, moisture, yellowing, disease...",
    description="Query:",
    style={"description_width": "60px"},
    layout=Layout(width="540px")
)

search_button = widgets.Button(description="Search", style=ButtonStyle(button_color="#2e7d5b"), layout=Layout(width="140px"))
search_output = Output()


def render_search_results(query=None, results=None, query_terms=None, answer=None, loading=False):
    """Render the full Search/RAG screen directly inside main_output."""
    display(HTML(APP_CSS))

    display(HTML("""
    <div class="card">
        <h2 class="section-title">🔍 Academic Search Engine + RAG</h2>
        <p class="muted">
            This screen searches the basil academic article index and sends
            the retrieved context to Gemini when the API key is configured.
        </p>
    </div>
    """))

    display(HBox([search_input, search_button]))

    if query is None:
        return

    display(HTML(f"""
    <div class="card">
        <h3>Search Results</h3>
        <p><b>Original query:</b> {html.escape(query)}</p>
        <p><b>Normalized query terms:</b> {html.escape(', '.join(query_terms or []))}</p>
        <p><b>Number of results:</b> {len(results or [])}</p>
        <p>{badge("Firebase connected" if FIREBASE_READY else "Firebase not connected", "green" if FIREBASE_READY else "orange")}</p>
        <p>{badge("Gemini ready" if GEMINI_MODEL else "Gemini fallback mode", "green" if GEMINI_MODEL else "orange")}</p>
    </div>
    """))

    if not results:
        display(HTML("""
        <div class="result-card">
            <h3>No matching articles found</h3>
            <p class="muted">Try terms such as mildew, disease, humidity, moisture, yellowing, leaf, or temperature.</p>
        </div>
        """))
    else:
        for article, score, matched_terms in results:
            snippet = article["text"][:450].replace("\n", " ")
            display(HTML(f"""
            <div class="result-card">
                <h3>{html.escape(article["title"])} <span class="muted">({article["id"]})</span></h3>
                <p><b>Authors:</b> {html.escape(article["authors"])}</p>
                <p><b>Score:</b> {score}</p>
                <p><b>Matched terms:</b> {html.escape(', '.join(matched_terms))}</p>
                <p>{html.escape(snippet)}...</p>
                <p class="muted">{html.escape(article["url"])}</p>
            </div>
            """))

    if loading:
        display(HTML("""
        <div class="rag-box">
            <h3>🧠 Gemini / RAG Answer</h3>
            <p>⏳ Generating answer, please wait...</p>
        </div>
        """))

    elif answer is not None:
        display(HTML(f"""
        <div class="rag-box">
            <h3>🧠 Gemini / RAG Answer</h3>
            <p>{html.escape(str(answer))}</p>
        </div>
        """))


def on_search_clicked(button):
    """Run search and refresh the whole search screen."""
    search_button.disabled = True
    search_button.description = "Searching..."

    query = search_input.value.strip()

    try:
        if not query:
            with main_output:
                clear_output(wait=True)
                render_search_results()
                display(HTML('<div class="card">Please type a search query first.</div>'))
            return

        results, query_terms = search_articles(query, ARTICLES, INVERTED_INDEX)

        # First refresh: show results + loading
        with main_output:
            clear_output(wait=True)
            render_search_results(query=query, results=results, query_terms=query_terms, loading=True)

        # Generate RAG answer
        answer = rag_answer(query, results)

        # Second refresh: show results + final answer
        with main_output:
            clear_output(wait=True)
            render_search_results(query=query, results=results, query_terms=query_terms, answer=answer)

    except Exception as e:
        with main_output:
            clear_output(wait=True)
            display(HTML(APP_CSS))
            display(HTML(f"""
            <div class="card">
                <h3>Search failed</h3>
                <p>{html.escape(str(e))}</p>
                <pre>{html.escape(traceback.format_exc())}</pre>
            </div>
            """))

    finally:
        search_button.disabled = False
        search_button.description = "Search"


def render_search_screen():
    """Search engine + RAG screen."""
    with main_output:
        clear_output(wait=True)
        render_search_results()


search_button.on_click(on_search_clicked)


In [ ]:
#@title #Main

dashboard_button = widgets.Button(
    description="📊 Dashboard",
    layout=Layout(width="190px", height="44px", margin="6px 0"),
    style=ButtonStyle(button_color="#2e7d5b")
)

search_nav_button = widgets.Button(
    description="🔍 Search / RAG",
    layout=Layout(width="190px", height="44px", margin="6px 0"),
    style=ButtonStyle(button_color="#ffffff")
)

upload_button = widgets.Button(
    description="📷 Upload Image",
    layout=Layout(width="190px", height="44px", margin="6px 0"),
    style=ButtonStyle(button_color="#ffffff")
)

iot_button = widgets.Button(
    description="📡 Sensors",
    layout=Layout(width="190px", height="44px", margin="6px 0"),
    style=ButtonStyle(button_color="#ffffff")
)


def set_active_button(active_button):
    """Highlight the selected navigation button."""
    for btn in [dashboard_button, upload_button, iot_button, search_nav_button]:
        btn.style.button_color = "#ffffff"
    active_button.style.button_color = "#2e7d5b"


def show_dashboard(button=None):
    set_active_button(dashboard_button)
    render_dashboard_screen()


def show_upload(button=None):
    set_active_button(upload_button)
    render_upload_screen()


def show_iot(button=None):
    set_active_button(iot_button)
    render_iot_screen()


def show_search(button=None):
    set_active_button(search_nav_button)
    render_search_screen()


dashboard_button.on_click(show_dashboard)
upload_button.on_click(show_upload)
iot_button.on_click(show_iot)
search_nav_button.on_click(show_search)

# Header

display(HTML("""
<div class="wb-app">
    <div class="wb-header">
        <h1>🌿 Wombat Basil Care</h1>
        <p>Academic Colab prototype for Firebase-based basil monitoring, academic search, and Gemini RAG.</p>
    </div>
</div>
"""))

# Left navigation + main screen area.
sidebar = VBox([
    widgets.HTML("<b style='color:#255f46'>Menu:</b>"),
    dashboard_button,
    search_nav_button,
    upload_button,
    iot_button
], layout=Layout(
    width="230px",
    min_width="230px",
    padding="10px 10px 10px 0"
))

main_output.layout = Layout(
    width="calc(100% - 250px)",
    min_width="0",
    overflow_x="auto"
)

app_body = HBox(
    [sidebar, main_output],
    layout=Layout(
        width="100%",
        align_items="flex-start"
    )
)

display(app_body)

# Open dashboard first by default.
show_dashboard()
